<a href="https://colab.research.google.com/github/Gr1lledChee5e/RegressionAnalysis/blob/main/Tree_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tree Based Methods**

Riley Yee

## Load PySpark

In [ ]:
#!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
#!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
#!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Tree Methods Example
  
We will be using a college dataset to try to classify colleges as Private or Public based off these features:

   **Private**: A factor with levels No and Yes indicating private or public university

   **Apps**: Number of applications received
  
  **Accept**: Number of applications accepted

  **Enroll**: Number of new students enrolled
  
   **Top10perc**: Pct. new students from top 10% of H.S. class

   **Top25perc**: Pct. new students from top 25% of H.S. class

   **F_Undergrad**: Number of fulltime undergraduates

   **P_Undergrad**: Number of parttime undergraduates

   **Outstate**: Out-of-state tuition

   **Room_Board:** Room and board costs

   **Books**: Estimated book costs

   **Personal**: Estimated personal spending

   **PhD**: Pct. of faculty with Ph.D.’s

   **Terminal**: Pct. of faculty with terminal degree

   **S_F_Ratio**: Student/faculty ratio

   **perc_alumni**: Pct. alumni who donate

   **Expend**: Instructional expenditure per student

   **Grad_Rate**: Graduation rate
    
Load in the data, print the schema, and display the data.

In [ ]:
data = spark.read.csv('/content/drive/MyDrive/Colab_Notebooks/ML_PySpark/College.csv', inferSchema=True, header=True)

In [ ]:
data.printSchema()

root
 |-- School: string (nullable = true)
 |-- Private: string (nullable = true)
 |-- Apps: integer (nullable = true)
 |-- Accept: integer (nullable = true)
 |-- Enroll: integer (nullable = true)
 |-- Top10perc: integer (nullable = true)
 |-- Top25perc: integer (nullable = true)
 |-- F_Undergrad: integer (nullable = true)
 |-- P_Undergrad: integer (nullable = true)
 |-- Outstate: integer (nullable = true)
 |-- Room_Board: integer (nullable = true)
 |-- Books: integer (nullable = true)
 |-- Personal: integer (nullable = true)
 |-- PhD: integer (nullable = true)
 |-- Terminal: integer (nullable = true)
 |-- S_F_Ratio: double (nullable = true)
 |-- perc_alumni: integer (nullable = true)
 |-- Expend: integer (nullable = true)
 |-- Grad_Rate: integer (nullable = true)



In [ ]:
data.show(truncate=0)

+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+
|School                                 |Private|Apps|Accept|Enroll|Top10perc|Top25perc|F_Undergrad|P_Undergrad|Outstate|Room_Board|Books|Personal|PhD|Terminal|S_F_Ratio|perc_alumni|Expend|Grad_Rate|
+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+
|Abilene Christian University           |Yes    |1660|1232  |721   |23       |52       |2885       |537        |7440    |3300      |450  |2200    |70 |78      |18.1     |12         |7041  |60       |
|Adelphi University                     |Yes    |2186|1924  |512   |16       |29       |2683       |1227       |12280   |6450      |750  |1500    |29 |30      |12.2     |16         |10527 |56       |


### Spark Formatting of Data

Print the columns of the data.

In [ ]:
data.columns

['School',
 'Private',
 'Apps',
 'Accept',
 'Enroll',
 'Top10perc',
 'Top25perc',
 'F_Undergrad',
 'P_Undergrad',
 'Outstate',
 'Room_Board',
 'Books',
 'Personal',
 'PhD',
 'Terminal',
 'S_F_Ratio',
 'perc_alumni',
 'Expend',
 'Grad_Rate']

We are interested in predicting whether a college is private or public. Therefore, column 'Private' is the response variable. Rest of the columns execpt the school name will be used as features

Create a vector/list of feature variable names

In [ ]:
feat = data.columns[2:]
print(feat)

['Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F_Undergrad', 'P_Undergrad', 'Outstate', 'Room_Board', 'Books', 'Personal', 'PhD', 'Terminal', 'S_F_Ratio', 'perc_alumni', 'Expend', 'Grad_Rate']


Import the vector assembler

In [ ]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=feat, outputCol='features')

Transform the data and view the output.

In [ ]:
output = assembler.transform(data)
output.show(truncate=0)

+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+----------------------------------------------------------------------------------------------------------+
|School                                 |Private|Apps|Accept|Enroll|Top10perc|Top25perc|F_Undergrad|P_Undergrad|Outstate|Room_Board|Books|Personal|PhD|Terminal|S_F_Ratio|perc_alumni|Expend|Grad_Rate|features                                                                                                  |
+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+----------------------------------------------------------------------------------------------------------+
|Abilene Christian University           |Yes    |1660|1232  |721   |23       |5

Deal with Private column being "Yes" or "No".

In [ ]:
from pyspark.ml.feature import StringIndexer

In [ ]:
indexer = StringIndexer(inputCol="Private", outputCol="PrivateIndex")

In [ ]:
output_fixed = indexer.fit(output).transform(output)

In [ ]:
output_fixed.show(truncate=0)

+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+----------------------------------------------------------------------------------------------------------+------------+
|School                                 |Private|Apps|Accept|Enroll|Top10perc|Top25perc|F_Undergrad|P_Undergrad|Outstate|Room_Board|Books|Personal|PhD|Terminal|S_F_Ratio|perc_alumni|Expend|Grad_Rate|features                                                                                                  |PrivateIndex|
+---------------------------------------+-------+----+------+------+---------+---------+-----------+-----------+--------+----------+-----+--------+---+--------+---------+-----------+------+---------+----------------------------------------------------------------------------------------------------------+------------+
|Abilene Christian University           

Split the data into 70% train and 30% test, use seed = 3.

In [ ]:
final_data = output_fixed.select("features",'PrivateIndex')

In [ ]:
train, test = final_data.randomSplit([0.7,0.3], seed=3)

Describe the train and test data sets.

In [ ]:
train.describe().show(truncate=0)
test.describe().show(truncate = 0)

+-------+-------------------+
|summary|PrivateIndex       |
+-------+-------------------+
|count  |516                |
|mean   |0.2868217054263566 |
|stddev |0.45271647741828214|
|min    |0.0                |
|max    |1.0                |
+-------+-------------------+

+-------+-------------------+
|summary|PrivateIndex       |
+-------+-------------------+
|count  |261                |
|mean   |0.24521072796934865|
|stddev |0.4310386088809289 |
|min    |0.0                |
|max    |1.0                |
+-------+-------------------+



### The Classifiers

Fit a decision tree, random forest and a gradient boosting classifier.

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier

In [ ]:
dtc = DecisionTreeClassifier(labelCol='PrivateIndex',featuresCol='features')

rfc = RandomForestClassifier(labelCol='PrivateIndex',featuresCol='features')

gbc = GBTClassifier(labelCol='PrivateIndex',featuresCol='features')

Train all three models:

In [ ]:
dtc_model = dtc.fit(train)

In [ ]:
dtc_model

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_15818fd42032, depth=5, numNodes=41, numClasses=2, numFeatures=17

In [ ]:
rfc_model = rfc.fit(train)

In [ ]:
gbc_model = gbc.fit(train)

## Model Comparison

Compare each of the models.

In [ ]:
dtc_predictions = dtc_model.transform(test)
rfc_predictions = rfc_model.transform(test)
gbc_predictions = gbc_model.transform(test)

In [ ]:
1/271

0.0036900369003690036

In [ ]:
dtc_predictions.show(truncate=0)

+------------------------------------------------------------------------------------------------------+------------+-------------+-----------------------------------------+----------+
|features                                                                                              |PrivateIndex|rawPrediction|probability                              |prediction|
+------------------------------------------------------------------------------------------------------+------------+-------------+-----------------------------------------+----------+
|[141.0,118.0,55.0,12.0,21.0,201.0,173.0,8300.0,4850.0,450.0,1300.0,53.0,53.0,9.5,19.0,6936.0,76.0]    |0.0         |[270.0,1.0]  |[0.996309963099631,0.0036900369003690036]|0.0       |
|[150.0,130.0,88.0,23.0,50.0,341.0,768.0,10300.0,4130.0,500.0,1700.0,44.0,58.0,10.2,37.0,9678.0,75.0]  |0.0         |[270.0,1.0]  |[0.996309963099631,0.0036900369003690036]|0.0       |
|[174.0,146.0,88.0,8.0,29.0,1047.0,33.0,8300.0,3080.0,600.0,600.0,62.0,62.0

Let's compare the models

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
model_accuracy = MulticlassClassificationEvaluator(labelCol='PrivateIndex', predictionCol='prediction', metricName='accuracy')

In [ ]:
dtc_accuracy = model_accuracy.evaluate(dtc_predictions)
rfc_accuracy = model_accuracy.evaluate(rfc_predictions)
gbc_accuracy = model_accuracy.evaluate(gbc_predictions)


In [ ]:
print('decision tree model accuracy is = ', dtc_accuracy, '\n')

print('random forest model accuracy is = ', rfc_accuracy, '\n')

print('gradient boosting model accuracy is = ', gbc_accuracy, '\n')

decision tree model accuracy is =  0.9272030651340997 

random forest model accuracy is =  0.9425287356321839 

gradient boosting model accuracy is =  0.9157088122605364 



## Let's calculate AUC values for all three tree based methods

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics

In [ ]:
my_eval = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction', labelCol='PrivateIndex')

In [ ]:
print('The AUC for decision tree model is = ', my_eval.evaluate(dtc_predictions))

The AUC for decision tree model is =  0.9074793781725887


In [ ]:
print('The AUC for random forest model is = ', my_eval.evaluate(rfc_predictions))

The AUC for random forest model is =  0.9877062182741116


In [ ]:
print('The AUC for gradient boosting model is = ', my_eval.evaluate(gbc_predictions))

The AUC for gradient boosting model is =  0.968789657360406
